# 01 Explore: South African electricity demand, April 2022 to October 2026

**Project:** forecast South Africa's hourly electricity demand one week ahead, and compare it with simple baselines and with Eskom's own published forecast.
**This notebook:** loads the Eskom data and documents the columns (Step 2), then explores trend, cycles, solar and load-shedding effects (Step 3).

**Data:** Eskom Data Portal ([eskom.co.za/dataportal](https://www.eskom.co.za/dataportal/)), requested through the [data request form](https://www.eskom.co.za/dataportal/data-request-form/) for the full available period (1 April 2022 to 31 March 2027). The download link arrives by email; the file is saved in `data/raw/`. Eskom's terms apply, so the raw file is not committed to this repository.

## 1. Load the data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

files = sorted(Path("../data/raw").glob("*.csv"))
if not files:
    raise FileNotFoundError("Request the data from https://www.eskom.co.za/dataportal/data-request-form/ "
                            "and save the CSV in data/raw/ (see README).")
raw = pd.read_csv(files[0])
print(files[0].name, raw.shape)
raw.head()

ESK19906.csv (43824, 17)


,Date Time Hour Beginning,Original Res Forecast before Lockdown,Residual Forecast,RSA Contracted Forecast,Dispatchable Generation,Residual Demand,RSA Contracted Demand,Wind,PV,CSP,Other RE,Total RE,Wind Installed Capacity,PV Installed Capacity,CSP Installed Capacity,Other RE Installed Capacity,Total RE Installed Capacity
0,2022-04-01 12:00:00 AM,22160.133,21324.592,22106.655,21105.801,21105.801,22240.789,1091.744,0.0,0.0,43.244,1134.988,3163.37,2212.09,500.0,50.58,5926.04
1,2022-04-01 01:00:00 AM,21737.774,20908.936,21651.713,20936.935,20936.935,22043.880,1063.786,0.0,0.0,43.159,1106.945,3163.37,2212.09,500.0,50.58,5926.04
2,2022-04-01 02:00:00 AM,21706.565,20771.960,21466.049,20892.068,20892.068,21880.597,956.343,0.0,0.0,32.186,988.529,3163.37,2212.09,500.0,50.58,5926.04
3,2022-04-01 03:00:00 AM,21940.481,20896.331,21560.271,20964.935,20964.935,21993.023,986.865,0.0,0.0,41.223,1028.088,3163.37,2212.09,500.0,50.58,5926.04
4,2022-04-01 04:00:00 AM,23045.826,21987.801,22620.957,21886.935,21886.935,22914.172,984.148,0.0,0.0,43.089,1027.237,3163.37,2212.09,500.0,50.58,5926.04


In [2]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 43824 entries, 0 to 43823
Data columns (total 17 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Date Time Hour Beginning               43824 non-null  str    
 1   Original Res Forecast before Lockdown  6600 non-null   float64
 2   Residual Forecast                      43824 non-null  float64
 3   RSA Contracted Forecast                43824 non-null  float64
 4   Dispatchable Generation                39600 non-null  float64
 5   Residual Demand                        39600 non-null  float64
 6   RSA Contracted Demand                  39600 non-null  float64
 7   Wind                                   39600 non-null  float64
 8   PV                                     39600 non-null  float64
 9   CSP                                    39600 non-null  float64
 10  Other RE                               39600 non-null  float64
 11  Total RE     

## 2. Parse the timestamps and check coverage

In [3]:
raw["timestamp"] = pd.to_datetime(raw["Date Time Hour Beginning"], format="%Y-%m-%d %I:%M:%S %p")
eskom = raw.drop(columns="Date Time Hour Beginning").set_index("timestamp")
steps = eskom.index.to_series().diff().value_counts()
print(f"From {eskom.index.min()} to {eskom.index.max()}, {len(eskom):,} hours")
print(f"Duplicated hours: {eskom.index.duplicated().sum()},  step sizes: {steps.to_dict()}")
print("\nNon-missing values per column:")
print(eskom.notna().sum().to_string())

From 2022-04-01 00:00:00 to 2027-03-31 23:00:00, 43,824 hours
Duplicated hours: 0,  step sizes: {Timedelta('0 days 01:00:00'): 43823}

Non-missing values per column:
Original Res Forecast before Lockdown     6600
Residual Forecast                        43824
RSA Contracted Forecast                  43824
Dispatchable Generation                  39600
Residual Demand                          39600
RSA Contracted Demand                    39600
Wind                                     39600
PV                                       39600
CSP                                      39600
Other RE                                 39600
Total RE                                 39600
Wind Installed Capacity                  39600
PV Installed Capacity                    39600
CSP Installed Capacity                   39600
Other RE Installed Capacity              39600
Total RE Installed Capacity              39600


In [4]:
last_actual = eskom["RSA Contracted Demand"].last_valid_index()
print(f"Actual demand recorded until: {last_actual}")
print(f"After that only Eskom's forecasts are filled in, up to {eskom.index.max().date()}.")
actual = eskom.loc[:last_actual]
print(f"Hours of actual data: {len(actual):,} ({len(actual) / 24 / 365.25:.1f} years), missing demand values: "
      f"{actual['RSA Contracted Demand'].isna().sum()}")

Actual demand recorded until: 2026-10-06 23:00:00
After that only Eskom's forecasts are filled in, up to 2027-03-31.
Hours of actual data: 39,600 (4.5 years), missing demand values: 0


## 3. What the columns mean

From the [Eskom Data Portal glossary](https://www.eskom.co.za/dataportal/glossary/) and checks on the data:

| Column | Meaning (MW, hourly average) |
|---|---|
| `RSA Contracted Demand` | Demand that must be supplied by **all resources Eskom has contracts with**, including renewable independent power producers. **This is the target.** |
| `Residual Demand` | Demand to be supplied by resources **Eskom National Control can dispatch**: contracted demand minus wind, solar and other renewables. |
| `RSA Contracted Forecast`, `Residual Forecast` | Eskom's own forecasts of the two demands. Filled in for every hour, including future hours up to March 2027. |
| `Original Res Forecast before Lockdown` | An older residual forecast series, only in the first months. Not used. |
| `Dispatchable Generation` | What the dispatchable fleet actually generated |
| `Wind`, `PV`, `CSP`, `Other RE`, `Total RE` | Contracted renewable generation: wind, solar photovoltaic, concentrated solar power, other (small hydro, biomass, landfill gas) |
| `... Installed Capacity` | Installed capacity of each renewable type |

Two checks: contracted demand minus residual demand should equal total renewables, and residual demand should equal dispatchable generation unless something stopped demand from being supplied.

In [5]:
diff = (actual["RSA Contracted Demand"] - actual["Residual Demand"] - actual["Total RE"]).abs()
print(f"Contracted - residual - renewables: median {diff.median():.0f} MW, 95th percentile {diff.quantile(0.95):.0f} MW")
gap = actual["Residual Demand"] - actual["Dispatchable Generation"]
print(f"Hours where residual demand exceeds dispatchable generation: {(gap > 1).mean():.0%}, largest gap {gap.max():,.0f} MW")

Contracted - residual - renewables: median 0 MW, 95th percentile 106 MW
Hours where residual demand exceeds dispatchable generation: 37%, largest gap 6,836 MW


- Contracted minus residual demand matches renewable generation closely, which confirms the definitions. (The small differences are other contracted sources not broken out in the file.)
- In 37% of hours, residual demand is higher than what the dispatchable fleet generated, by up to 6,800 MW. Those hours cluster in 2022 and 2023, the years of heavy **load shedding**. The demand columns appear to include demand that was cut off, which is what a planner needs to forecast: how much the country **wants**, not only what it got. Step 3 looks at this in time.